In [ ]:
import os
import gc
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from dateutil import parser

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer  

In [ ]:

# =====================
# CONFIG
# =====================
RAW_FOLDER = "/home/sunkari/Stock_price_predictor/Dataset"
OUTPUT_FOLDER = "./processed_datasets"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Use GPU only if available and mostly free. Otherwise CPU.
USE_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if USE_CUDA else "cpu")
print("Device:", DEVICE)

# =====================
# Load models safely
# =====================
# Lightweight sentence-transformer for embeddings (fast + memory friendly)
EMB_MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"  # 768-dim
# Finance sentiment classifier
SENT_MODEL_NAME = "yiyanghkust/finbert-tone"

# Load embedding model (SentenceTransformer handles device internally)
print("Loading embedding model:", EMB_MODEL_NAME)
emb_model = SentenceTransformer(EMB_MODEL_NAME, device=str(DEVICE))  # will be on CPU or GPU
emb_model.max_seq_length = 128

print("Loading sentiment model:", SENT_MODEL_NAME)
sent_tokenizer = AutoTokenizer.from_pretrained(SENT_MODEL_NAME)
sent_model = AutoModelForSequenceClassification.from_pretrained(SENT_MODEL_NAME)
# Put sentiment model on CPU by default, or GPU if you have enough free memory:
try:
    if USE_CUDA:
        # cautiously try GPU
        sent_model.to(DEVICE)
        print("Sentiment model moved to GPU.")
    else:
        sent_model.to("cpu")
except RuntimeError:
    # fallback to CPU
    torch.cuda.empty_cache()
    sent_model.to("cpu")
    DEVICE = torch.device("cpu")
    print("Fell back to CPU for sentiment model.")

sent_model.eval()

# =====================
# Utilities
# =====================
def normalize_date_column(df):
    def parse_date_safe(x):
        try:
            return parser.parse(str(x), dayfirst=False)
        except:
            try:
                return parser.parse(str(x), dayfirst=True)
            except:
                return pd.NaT
    df["Date"] = df["Date"].apply(parse_date_safe)
    df = df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)
    return df

def split_headlines(text):
    if pd.isna(text):
        return []
    return [t.strip() for t in str(text).split('|') if t.strip()]

# batch inference helpers
@torch.inference_mode()
def compute_sentiment_scores(texts, batch_size=16, device=DEVICE):
    """Return array of sentiment scores (positive_prob - negative_prob) for each text."""
    scores = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = sent_tokenizer(batch, padding=True, truncation=True, return_tensors="pt", max_length=128)
        # move tensors to device where model is
        model_device = next(sent_model.parameters()).device
        inputs = {k: v.to(model_device) for k, v in inputs.items()}
        out = sent_model(**inputs)
        probs = torch.nn.functional.softmax(out.logits, dim=-1).cpu().numpy()  # shape (B, 3)
        # finbert-tone ordering: check model card, but common: [neg, neu, pos]
        # score = pos - neg
        batch_scores = probs[:, 2] - probs[:, 0]
        scores.extend(batch_scores.tolist())
        # free GPU
        if model_device.type == "cuda":
            torch.cuda.empty_cache()
            gc.collect()
    return np.array(scores, dtype=np.float32)

def compute_embeddings(texts, batch_size=32):
    """
    Use sentence-transformers model which handles batching internally.
    Returns array shape (len(texts), emb_dim)
    """
    # SentenceTransformer.encode supports batching and returns numpy array
    embs = emb_model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=False,
        convert_to_numpy=True,
        max_length=128,
        truncation=True
    )
    return embs  # (N, D)

# =====================
# Main loop
# =====================
files = [f for f in os.listdir(RAW_FOLDER) if f.endswith(".csv")]
for fname in tqdm(files, desc="Processing CSV files"):
    path = os.path.join(RAW_FOLDER, fname)
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip().str.replace('\ufeff', '', regex=False)
    df = normalize_date_column(df)
    if "Headlines" not in df.columns:
        print(f"Skipping {fname}: no Headlines column")
        continue

    df["Headline_List"] = df["Headlines"].apply(split_headlines)

    all_embs = []
    all_sentiments = []

    # We'll collect all headline texts for batch encoding / sentiment, but per-row groups vary.
    # Approach: flatten (row_idx, headline) pairs -> compute embeddings and sentiment for all headlines -> group back.
    row_headlines = []   # flat list of headline strings
    row_index_map = []   # maps each headline to row idx

    for idx, hl_list in enumerate(df["Headline_List"].tolist()):
        if not hl_list:
            continue
        for h in hl_list:
            row_headlines.append(h)
            row_index_map.append(idx)

    if len(row_headlines) == 0:
        # no headlines at all; fill zeros
        emb_dim = emb_model.get_sentence_embedding_dimension()
        zeros = np.zeros((len(df), emb_dim), dtype=np.float32)
        df_out = pd.concat([df.drop(columns=["Headline_List"]), pd.DataFrame(zeros, columns=[f"emb_{i}" for i in range(zeros.shape[1])])], axis=1)
        df_out["sentiment_score"] = 0.0
        out_path = os.path.join(OUTPUT_FOLDER, fname.replace(".csv", "_merged.csv"))
        df_out.to_csv(out_path, index=False)
        continue

    # Compute embeddings for all headlines in batches (memory friendly)
    headline_embeddings = compute_embeddings(row_headlines, batch_size=32)  # shape (total_headlines, emb_dim)

    # Compute sentiment scores for all headlines (uses sentiment model)
    headline_sentiments = compute_sentiment_scores(row_headlines, batch_size=32)  # shape (total_headlines,)

    # Now aggregate per-row: mean embedding and mean sentiment across headlines for that row
    emb_dim = headline_embeddings.shape[1]
    per_row_embs = np.zeros((len(df), emb_dim), dtype=np.float32)
    per_row_sents = np.zeros(len(df), dtype=np.float32)
    counts = np.zeros(len(df), dtype=np.int32)

    for i, row_idx in enumerate(row_index_map):
        per_row_embs[row_idx] += headline_embeddings[i]
        per_row_sents[row_idx] += headline_sentiments[i]
        counts[row_idx] += 1

    nonzero = counts > 0
    per_row_embs[nonzero] = per_row_embs[nonzero] / counts[nonzero][:, None]
    per_row_sents[nonzero] = per_row_sents[nonzero] / counts[nonzero]

    # For rows without headlines, keep zeros and sentiment 0.0 (neutral)
    emb_cols = [f"emb_{i}" for i in range(emb_dim)]
    emb_df = pd.DataFrame(per_row_embs, columns=emb_cols)
    df["sentiment_score"] = per_row_sents

    df_out = pd.concat([df.drop(columns=["Headline_List"]), emb_df], axis=1)
    out_path = os.path.join(OUTPUT_FOLDER, fname.replace(".csv", "_merged.csv"))
    df_out.to_csv(out_path, index=False)

    # cleanup per-file
    del headline_embeddings, headline_sentiments, row_headlines, row_index_map
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Done.")

## final generation

In [ ]:
import os
import time
import logging
import torch
import torch.nn.functional as F
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from dateutil import parser

# =====================================================
LOG_FILE = "processing_log.txt"
logging.basicConfig(
    filename=LOG_FILE,
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

def log(msg, level="info"):
    tqdm.write(msg)
    if level == "error":
        logging.error(msg)
        
    elif level == "warning":
        logging.warning(msg)

    else:
        logging.info(msg)

# =====================================================

def normalize_date_column(df):

    def parse_date_safe(x):
        try:
            return parser.parse(str(x), dayfirst=False)

        except Exception:
            try:
                return parser.parse(str(x), dayfirst=True)
            except Exception:
                return None

    df["Date"] = df["Date"].apply(parse_date_safe)
    df = df.dropna(subset=["Date"])
    df = df.sort_values("Date").reset_index(drop=True)
    return df

def split_headlines(text):
    if pd.isna(text):
        return []
    return [t.strip() for t in str(text).split('|') if t.strip()]

# =====================================================

DATA_DIR = "/home/sunkari/Stock_price_predictor/Dataset" 
OUTPUT_DIR = "./Processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)
TRAIN_RATIO = 0.8  # 80% train, 20% test

# =====================================================

MODEL_NAME = "yiyanghkust/finbert-tone"
log(f"🔹 Loading model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.eval()
log("✅ Model loaded successfully.")

# =====================================================

def get_sentiment_scores(texts):

    if len(texts) == 0:
        return [0.0, 1.0, 0.0]  # Neutral default

    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1)
    return probs.mean(dim=0).numpy().tolist()
# =====================================================

start_time_total = time.time()
for file in os.listdir(DATA_DIR):

    if not file.endswith(".csv"):
        continue

    start_time = time.time()
    company_path = os.path.join(DATA_DIR, file)
    log(f"\n🔍 Processing {file} ...")

    try:
        df = pd.read_csv(company_path)
        df.columns = df.columns.str.strip()
        log(f"📂 Loaded file with {len(df)} rows.")

        df = normalize_date_column(df)
        log(f"🗓️ Normalized dates, {len(df)} rows remain after cleaning.")

        # Split headlines into lists
        df["Headline_List"] = df["Headlines"].apply(split_headlines)

        # Compute daily sentiment
        sentiments = []

        for headlines in tqdm(df["Headline_List"].tolist(), desc=f"Sentiment {file}"):
            try:
                probs = get_sentiment_scores(headlines)
                sentiments.append(probs)

            except Exception as e:

                log(f"⚠️ Error processing headlines: {headlines[:3]}... | {e}", "warning")
                sentiments.append([0.0, 1.0, 0.0])  # default neutral

        sentiments = pd.DataFrame(sentiments, columns=["negative", "neutral", "positive"])

        df = pd.concat([df, sentiments], axis=1)
        # --- 🧹 CLEANUP (As Requested) ---

        # Drop the text columns after processing
        df = df.drop(columns=["Headlines", "Headline_List"], errors='ignore')
        
        # Save processed versions
        base_name = os.path.splitext(file)[0]
        train_out = os.path.join(OUTPUT_DIR, f"{base_name}_train.csv")
        df.to_csv(train_out, index=False)
        elapsed = time.time() - start_time
        log(f"✅ Saved data split for {file} | data={len(df)} | ⏱️ {elapsed:.2f}s")



    except Exception as e:

        log(f"❌ Error processing {file}: {e}", "error")



total_time = time.time() - start_time_total

log(f"\n🏁 All files processed successfully in {total_time/60:.2f} minutes.")

🔹 Loading model: yiyanghkust/finbert-tone
✅ Model loaded successfully.

🔍 Processing XOM_stock_gdelt_final.csv ...
📂 Loaded file with 1003 rows.
🗓️ Normalized dates, 1003 rows remain after cleaning.


Sentiment XOM_stock_gdelt_final.csv: 100%|██████████| 1003/1003 [01:03<00:00, 15.78it/s]


✅ Saved data split for XOM_stock_gdelt_final.csv | data=1003 | ⏱️ 63.66s

🔍 Processing MSFT_stock_gdelt_final.csv ...
📂 Loaded file with 1003 rows.
🗓️ Normalized dates, 1003 rows remain after cleaning.


Sentiment MSFT_stock_gdelt_final.csv: 100%|██████████| 1003/1003 [01:36<00:00, 10.44it/s]


✅ Saved data split for MSFT_stock_gdelt_final.csv | data=1003 | ⏱️ 96.16s

🔍 Processing V_stock_gdelt_final.csv ...
📂 Loaded file with 1003 rows.
🗓️ Normalized dates, 1003 rows remain after cleaning.


...

✅ Saved data split for TSLA_stock_gdelt_final.csv | data=1003 | ⏱️ 105.81s

🔍 Processing JPM_stock_gdelt_final.csv ...
📂 Loaded file with 1003 rows.
🗓️ Normalized dates, 1003 rows remain after cleaning.


Sentiment JPM_stock_gdelt_final.csv: 100%|██████████| 1003/1003 [01:42<00:00,  9.81it/s]


✅ Saved data split for JPM_stock_gdelt_final.csv | data=1003 | ⏱️ 102.40s

🔍 Processing AAPL_stock_gdelt_final.csv ...
📂 Loaded file with 1003 rows.
🗓️ Normalized dates, 1003 rows remain after cleaning.


Sentiment AAPL_stock_gdelt_final.csv: 100%|██████████| 1003/1003 [01:23<00:00, 12.06it/s]

✅ Saved data split for AAPL_stock_gdelt_final.csv | data=1003 | ⏱️ 83.26s

🏁 All files processed successfully in 14.80 minutes.


In [ ]:
import os
import glob
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import pickle
from tqdm import tqdm

# ------------------------------
# CONFIGURATION
# ------------------------------
INPUT_DIR = "Stock_price_predictor/Processed"
OUTPUT_DIR = "windows"
SCALER_DIR = "scalers"
WINDOW_SIZE = 8
TARGET_COLUMN = "Close"
TEST_SPLIT_RATIO = 0.2

# Columns to normalize
NUMERIC_COLS = [
    "Adj Close", "Close", "High", "Low", "Open",
    "Volume", "Daily_Return", "EMA_7", "EMA_21"
]
# Sentiment columns (not normalized)
SENTIMENT_COLS = ["negative", "neutral", "positive"]

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SCALER_DIR, exist_ok=True)

print("📂 Building train/test windows including sentiment columns...")

# ------------------------------
# LOAD & PROCESS ALL FILES
# ------------------------------
train_data, test_data = [], []
company_list = []

for file_path in tqdm(glob.glob(os.path.join(INPUT_DIR, "*.csv"))):
    company_name = os.path.splitext(os.path.basename(file_path))[0]
    company_list.append(company_name)

    df = pd.read_csv(file_path)
    df = df.dropna(subset=[TARGET_COLUMN])
    df = df.reset_index(drop=True)

    # Ensure required columns exist
    if not all(col in df.columns for col in NUMERIC_COLS + SENTIMENT_COLS):
        print(f"⚠️ Skipping {company_name} — missing columns.")
        continue

    # Split train/test
    split_idx = int(len(df) * (1 - TEST_SPLIT_RATIO))
    df_train, df_test = df.iloc[:split_idx], df.iloc[split_idx:]

    # Fit scaler on train numeric columns
    scaler = MinMaxScaler()
    scaler.fit(df_train[NUMERIC_COLS])

    # Save scaler for denormalization later
    with open(os.path.join(SCALER_DIR, f"{company_name}_scaler.pkl"), "wb") as f:
        pickle.dump(scaler, f)

    # Normalize numeric columns only
    for subset, name in [(df_train, "train"), (df_test, "test")]:
        numeric_scaled = scaler.transform(subset[NUMERIC_COLS])
        sentiment_values = subset[SENTIMENT_COLS].values  # unchanged
        combined = np.concatenate([numeric_scaled, sentiment_values], axis=1)

        # Build windows
        for i in range(len(combined) - WINDOW_SIZE):
            X = combined[i:i+WINDOW_SIZE]
            y = subset[TARGET_COLUMN].iloc[i + WINDOW_SIZE]  # normalized target
            if name == "train":
                train_data.append((X, y, company_name))
            else:
                test_data.append((X, y, company_name))

print(f"✅ Train windows: {len(train_data)} | Test windows: {len(test_data)} | Companies: {len(company_list)}")

# ------------------------------
# SAVE OUTPUTS
# ------------------------------
pickle.dump(train_data, open(os.path.join(OUTPUT_DIR, "train_windows.pkl"), "wb"))
pickle.dump(test_data, open(os.path.join(OUTPUT_DIR, "test_windows.pkl"), "wb"))
pickle.dump(company_list, open(os.path.join(OUTPUT_DIR, "train_company_list.pkl"), "wb"))

print(f"✅ Saved pickles to '{OUTPUT_DIR}' and scalers to '{SCALER_DIR}'")

# ------------------------------
# Summary of feature layout
# ------------------------------
print("\n🧾 Feature layout per sample:")
for i, col in enumerate(NUMERIC_COLS + SENTIMENT_COLS):
    print(f"{i:2d}. {col}")
print("\n✅ Preprocessing complete.")

📂 Building train/test windows including sentiment columns...


0it [00:00, ?it/s]

✅ Train windows: 0 | Test windows: 0 | Companies: 0
✅ Saved pickles to 'windows' and scalers to 'scalers'

🧾 Feature layout per sample:
 0. Adj Close
 1. Close
 2. High
 3. Low
 4. Open
 5. Volume
 6. Daily_Return
 7. EMA_7
 8. EMA_21
 9. negative
10. neutral
11. positive

✅ Preprocessing complete.
